In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # Conflit de versions entre Keras3 et tf hub

In [2]:
import pandas as pd
import numpy as np
import re
import string

import plotly.graph_objects as go
import plotly.io as pio
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, Dropout

c:\Users\Korvydae\anaconda3\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



In [3]:
BG_COLOR = "#262626"
COLOR_RED = "#b91c1c"
COLOR_PINK = "#db2777"
pio.templates.default = "plotly_dark"

def style(fig):
    fig.update_layout(
        paper_bgcolor=BG_COLOR,
        plot_bgcolor=BG_COLOR,
        font_color="#ffffff",
        margin=dict(t=50, b=50, l=50, r=50)
    )
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(gridcolor="#333333")
    return fig

## Chargement des données

In [7]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'text']
df.head()


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## EDA

In [31]:
# Comptage des occurrences
counts = df['label'].value_counts().reset_index()
counts.columns = ['label', 'count']

fig = px.pie(
    counts, 
    values='count', 
    names='label',
    color='label',
    color_discrete_map={'ham': COLOR_PINK, 'spam': COLOR_RED},
    hole=0.4
)

fig.update_traces(
    textinfo='percent+label', 
    showlegend=False,
    marker=dict(line=dict(color=BG_COLOR, width=2))
)

fig.update_layout(
    title={
        'text': "Répartition des messages", 'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    showlegend=False
)

fig = style(fig)
fig.show()

# AVEC NETTOYAGE

## Prétraitement

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

y = df['label_num'].values

X_text_train, X_text_test, y_train, y_test = train_test_split(
    df['clean_text'].values, y, test_size=0.2, random_state=42
)

# Universal Sentence Encoder pour avoir un modèle pré-entraîné
url = "https://tfhub.dev/google/universal-sentence-encoder/4"

## Construction du modèle

In [6]:
input_layer = Input(shape=(), dtype=tf.string)
use_layer = hub.KerasLayer(url, trainable=False)(input_layer)

x = Dense(64, activation='relu')(use_layer)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model_clean = Model(inputs=input_layer, outputs=output)

model_clean.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_clean.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None,)]                 0         
                                                                 
 keras_layer (KerasLayer)    (None, 512)               256797824 
                                                                 
 dense (Dense)               (None, 64)                32832     
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 256830721 (979.73 MB)
Trainable params: 32897 (128.50 KB)
Non-trainable params: 256797824 (979.61 MB)
_________________________________________________________________


## Entrainement

In [7]:
# Calcul automatique des poids selon la fréquence de chaque classe
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print("Poids des classes :")
print(f"  HAM  (0) : {class_weight_dict[0]:.4f}")
print(f"  SPAM (1) : {class_weight_dict[1]:.4f}")

history_clean = model_clean.fit(
    X_text_train, y_train,
    epochs=10,
    validation_data=(X_text_test, y_test),
    class_weight=class_weight_dict,
    verbose=1
)

# Prédictions
y_pred_clean = (model_clean.predict(X_text_test) > 0.5).astype("int32")
print(classification_report(y_test, y_pred_clean, target_names=["HAM", "SPAM"]))

Poids des classes :
  HAM  (0) : 0.5773
  SPAM (1) : 3.7328
Epoch 1/10



140/140 [==============================] - 4s 11ms/step - loss: 0.3785 - accuracy: 0.9147 - val_loss: 0.1595 - val_accuracy: 0.9677
Epoch 2/10
140/140 [==============================] - 1s 8ms/step - loss: 0.1342 - accuracy: 0.9666 - val_loss: 0.1048 - val_accuracy: 0.9686
Epoch 3/10
140/140 [==============================] - 1s 8ms/step - loss: 0.1003 - accuracy: 0.9720 - val_loss: 0.0929 - val_accuracy: 0.9722
Epoch 4/10
140/140 [==============================] - 1s 8ms/step - loss: 0.0833 - accuracy: 0.9776 - val_loss: 0.1003 - val_accuracy: 0.9659
Epoch 5/10
140/140 [==============================] - 1s 7ms/step - loss: 0.0715 - accuracy: 0.9785 - val_loss: 0.0871 - val_accuracy: 0.9722
Epoch 6/10
140/140 [==============================] - 1s 7ms/step - loss: 0.0619 - accuracy: 0.9789 - val_loss: 0.0758 - val_accuracy: 0.9740
Epoch 7/10
140/140 [==============================] - 1s 8ms/step - loss: 0.0573 - accuracy: 0.9825 - val_loss: 0.0753 - val_accuracy: 0.9758
Epoch 8/10
140/1

## Performances

In [8]:
acc = history_clean.history['accuracy']
val_acc = history_clean.history['val_accuracy']
loss = history_clean.history['loss']
val_loss = history_clean.history['val_loss']
epochs = list(range(1, len(acc) + 1))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Accuracy', 'Loss'))

# Accuracy
fig.add_trace(go.Scatter(x=epochs, y=acc, name='Train Accuracy', line=dict(color=COLOR_PINK)), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=val_acc, name='Val Accuracy', line=dict(color=COLOR_RED, dash='dash')), row=1, col=1)

# Loss
fig.add_trace(go.Scatter(x=epochs, y=loss, name='Train Loss', line=dict(color=COLOR_PINK)), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs, y=val_loss, name='Val Loss', line=dict(color=COLOR_RED, dash='dash')), row=1, col=2)

style(fig)
fig.show()

In [9]:
cm = confusion_matrix(y_test, y_pred_clean)

fig = ff.create_annotated_heatmap(
    z=cm,
    x=['Prédit HAM', 'Prédit SPAM'],
    y=['Réel HAM', 'Réel SPAM'],
    colorscale=[[0, BG_COLOR], [0.5, COLOR_RED], [1, COLOR_PINK]],
    annotation_text=cm,
    showscale=False
)

fig.update_layout(
    title='Matrice de confusion',
    height=400,
    width=400
)

style(fig)
fig.show()

In [10]:
# Résultats
accuracy_clean = accuracy_score(y_test, y_pred_clean)
f1_clean = f1_score(y_test, y_pred_clean)

print(f"Accuracy  : {accuracy_clean:.2%}")
print(f"F1-Score  : {f1_clean:.4f}")
print(f"Loss finale (train)      : {history_clean.history['loss'][-1]:.4f}")
print(f"Loss finale (validation) : {history_clean.history['val_loss'][-1]:.4f}")
print(f"Accuracy finale (train)      : {history_clean.history['accuracy'][-1]:.2%}")
print(f"Accuracy finale (validation) : {history_clean.history['val_accuracy'][-1]:.2%}")
print("Classification Report :")
print(classification_report(y_test, y_pred_clean, target_names=["HAM", "SPAM"]))

Accuracy  : 97.76%
F1-Score  : 0.9196
Loss finale (train)      : 0.0426
Loss finale (validation) : 0.0749
Accuracy finale (train)      : 98.56%
Accuracy finale (validation) : 97.76%
Classification Report :
              precision    recall  f1-score   support

         HAM       0.99      0.98      0.99       965
        SPAM       0.89      0.95      0.92       150

    accuracy                           0.98      1115
   macro avg       0.94      0.97      0.95      1115
weighted avg       0.98      0.98      0.98      1115



## Prédictions

In [11]:
def predict_spam(new_sms):
    cleaned = clean_text(new_sms)
    prediction = model_clean.predict([cleaned])[0][0]
    label = "SPAM 🚨" if prediction > 0.5 else "HAM ✅"
    print(f"Message: {new_sms} \nRésultat: {label} (Confiance: {prediction:.2%})\n")

# Tests
# SPAMS
predict_spam("URGENT! Your mobile number has been selected to win a £2000 prize. Call 09061743810 now!")
predict_spam("FREE entry to our weekly competition! Text WIN to 80086 and claim your reward.")
predict_spam("You have been pre-approved for a £5000 loan. No credit checks. Reply YES to apply.")
predict_spam("Congratulations! You are our lucky winner this week. Click http://claim-prize.com now.")
predict_spam("Your account has been suspended. Verify your identity immediately at secure-login.net")
# HAMS
predict_spam("Hey, can you pick up some milk on your way home?")
predict_spam("Don't forget we have a meeting at 3pm tomorrow.")
predict_spam("Happy birthday! Hope you have an amazing day 🎉")
predict_spam("I'm running 10 minutes late, save me a seat!")
predict_spam("Did you watch the game last night? Unbelievable ending.")

1/1 [==============================] - 0s 34ms/step
Message: URGENT! Your mobile number has been selected to win a £2000 prize. Call 09061743810 now! 
Résultat: SPAM 🚨 (Confiance: 100.00%)

1/1 [==============================] - 0s 31ms/step
Message: FREE entry to our weekly competition! Text WIN to 80086 and claim your reward. 
Résultat: SPAM 🚨 (Confiance: 99.93%)

1/1 [==============================] - 0s 34ms/step
Message: You have been pre-approved for a £5000 loan. No credit checks. Reply YES to apply. 
Résultat: SPAM 🚨 (Confiance: 96.83%)

1/1 [==============================] - 0s 31ms/step
Message: Congratulations! You are our lucky winner this week. Click http://claim-prize.com now. 
Résultat: SPAM 🚨 (Confiance: 98.27%)

1/1 [==============================] - 0s 33ms/step
Message: Your account has been suspended. Verify your identity immediately at secure-login.net 
Résultat: SPAM 🚨 (Confiance: 98.36%)

1/1 [==============================] - 0s 33ms/step
Message: Hey, can you p

# SANS NETTOYAGE

## Prétraitement

In [12]:
# Sans nettoyage du texte, moins  utile avec USE et moins pertinent pour détecter les spams

df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

y = df['label_num'].values

X_text_train, X_text_test, y_train, y_test = train_test_split(
    df['text'].values, y, test_size=0.2, random_state=42
)

# Universal Sentence Encoder pour avoir un modèle pré-entraîné
url = "https://tfhub.dev/google/universal-sentence-encoder/4"

## Construction du modèle

In [13]:
input_layer = Input(shape=(), dtype=tf.string)
use_layer = hub.KerasLayer(url, trainable=False)(input_layer)

x = Dense(64, activation='relu')(use_layer)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model_raw = Model(inputs=input_layer, outputs=output)

model_raw.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_raw.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None,)]                 0         
                                                                 
 keras_layer_1 (KerasLayer)  (None, 512)               256797824 
                                                                 
 dense_2 (Dense)             (None, 64)                32832     
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_3 (Dense)             (None, 1)                 65        
                                                                 
Total params: 256830721 (979.73 MB)
Trainable params: 32897 (128.50 KB)
Non-trainable params: 256797824 (979.61 MB)
_________________________________________________________________


## Entrainement

In [14]:
# Calcul automatique des poids selon la fréquence de chaque classe
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print("Poids des classes :")
print(f"  HAM  (0) : {class_weight_dict[0]:.4f}")
print(f"  SPAM (1) : {class_weight_dict[1]:.4f}")

history_raw = model_raw.fit(
    X_text_train, y_train,
    epochs=10,
    validation_data=(X_text_test, y_test),
    class_weight=class_weight_dict,
    verbose=1
)

# Prédictions
y_pred_raw = (model_raw.predict(X_text_test) > 0.5).astype("int32")
print(classification_report(y_test, y_pred_raw, target_names=["HAM", "SPAM"]))

Poids des classes :
  HAM  (0) : 0.5773
  SPAM (1) : 3.7328
Epoch 1/10
140/140 [==============================] - 3s 10ms/step - loss: 0.3341 - accuracy: 0.8905 - val_loss: 0.1497 - val_accuracy: 0.9695
Epoch 2/10
140/140 [==============================] - 1s 8ms/step - loss: 0.1138 - accuracy: 0.9735 - val_loss: 0.0945 - val_accuracy: 0.9776
Epoch 3/10
140/140 [==============================] - 1s 8ms/step - loss: 0.0847 - accuracy: 0.9764 - val_loss: 0.0806 - val_accuracy: 0.9794
Epoch 4/10
140/140 [==============================] - 1s 8ms/step - loss: 0.0666 - accuracy: 0.9816 - val_loss: 0.0811 - val_accuracy: 0.9803
Epoch 5/10
140/140 [==============================] - 1s 8ms/step - loss: 0.0548 - accuracy: 0.9832 - val_loss: 0.0645 - val_accuracy: 0.9839
Epoch 6/10
140/140 [==============================] - 1s 8ms/step - loss: 0.0506 - accuracy: 0.9838 - val_loss: 0.0723 - val_accuracy: 0.9803
Epoch 7/10
140/140 [==============================] - 1s 8ms/step - loss: 0.0432 - accu

## Performances

In [15]:
acc = history_raw.history['accuracy']
val_acc = history_raw.history['val_accuracy']
loss = history_raw.history['loss']
val_loss = history_raw.history['val_loss']
epochs = list(range(1, len(acc) + 1))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Accuracy', 'Loss'))

# Accuracy
fig.add_trace(go.Scatter(x=epochs, y=acc, name='Train Accuracy', line=dict(color=COLOR_PINK)), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=val_acc, name='Val Accuracy', line=dict(color=COLOR_RED, dash='dash')), row=1, col=1)

# Loss
fig.add_trace(go.Scatter(x=epochs, y=loss, name='Train Loss', line=dict(color=COLOR_PINK)), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs, y=val_loss, name='Val Loss', line=dict(color=COLOR_RED, dash='dash')), row=1, col=2)

style(fig)
fig.show()

In [16]:
cm = confusion_matrix(y_test, y_pred_raw)

fig = ff.create_annotated_heatmap(
    z=cm,
    x=['Prédit HAM', 'Prédit SPAM'],
    y=['Réel HAM', 'Réel SPAM'],
    colorscale=[[0, BG_COLOR], [0.5, COLOR_RED], [1, COLOR_PINK]],
    annotation_text=cm,
    showscale=False
)

fig.update_layout(
    title='Matrice de confusion',
    height=400,
    width=400
)

style(fig)
fig.show()

In [17]:
# Résultats
accuracy_raw = accuracy_score(y_test, y_pred_raw)
f1_raw = f1_score(y_test, y_pred_raw)

print(f"Accuracy  : {accuracy_raw:.2%}")
print(f"F1-Score  : {f1_raw:.4f}")
print(f"Loss finale (train)      : {history_raw.history['loss'][-1]:.4f}")
print(f"Loss finale (validation) : {history_raw.history['val_loss'][-1]:.4f}")
print(f"Accuracy finale (train)      : {history_raw.history['accuracy'][-1]:.2%}")
print(f"Accuracy finale (validation) : {history_raw.history['val_accuracy'][-1]:.2%}")
print("Classification Report :")
print(classification_report(y_test, y_pred_raw, target_names=["HAM", "SPAM"]))

Accuracy  : 98.12%
F1-Score  : 0.9320
Loss finale (train)      : 0.0331
Loss finale (validation) : 0.0639
Accuracy finale (train)      : 98.95%
Accuracy finale (validation) : 98.12%
Classification Report :
              precision    recall  f1-score   support

         HAM       0.99      0.98      0.99       965
        SPAM       0.91      0.96      0.93       150

    accuracy                           0.98      1115
   macro avg       0.95      0.97      0.96      1115
weighted avg       0.98      0.98      0.98      1115



## Prédictions

In [18]:
def predict_spam(new_sms):
    prediction = model_raw.predict([new_sms])[0][0]
    label = "SPAM 🚨" if prediction > 0.5 else "HAM ✅"
    print(f"Message: {new_sms} \nRésultat: {label} (Confiance: {prediction:.2%})\n")

# Tests
# SPAMS
predict_spam("URGENT! Your mobile number has been selected to win a £2000 prize. Call 09061743810 now!")
predict_spam("FREE entry to our weekly competition! Text WIN to 80086 and claim your reward.")
predict_spam("You have been pre-approved for a £5000 loan. No credit checks. Reply YES to apply.")
predict_spam("Congratulations! You are our lucky winner this week. Click http://claim-prize.com now.")
predict_spam("Your account has been suspended. Verify your identity immediately at secure-login.net")
# HAMS
predict_spam("Hey, can you pick up some milk on your way home?")
predict_spam("Don't forget we have a meeting at 3pm tomorrow.")
predict_spam("Happy birthday! Hope you have an amazing day 🎉")
predict_spam("I'm running 10 minutes late, save me a seat!")
predict_spam("Did you watch the game last night? Unbelievable ending.")

1/1 [==============================] - 0s 47ms/step
Message: URGENT! Your mobile number has been selected to win a £2000 prize. Call 09061743810 now! 
Résultat: SPAM 🚨 (Confiance: 100.00%)

1/1 [==============================] - 0s 42ms/step
Message: FREE entry to our weekly competition! Text WIN to 80086 and claim your reward. 
Résultat: SPAM 🚨 (Confiance: 99.94%)

1/1 [==============================] - 0s 33ms/step
Message: You have been pre-approved for a £5000 loan. No credit checks. Reply YES to apply. 
Résultat: SPAM 🚨 (Confiance: 82.08%)

1/1 [==============================] - 0s 31ms/step
Message: Congratulations! You are our lucky winner this week. Click http://claim-prize.com now. 
Résultat: SPAM 🚨 (Confiance: 95.72%)

1/1 [==============================] - 0s 32ms/step
Message: Your account has been suspended. Verify your identity immediately at secure-login.net 
Résultat: SPAM 🚨 (Confiance: 95.38%)

1/1 [==============================] - 0s 33ms/step
Message: Hey, can you p

# CONCLUSION

In [26]:
print(f"{'USE + texte nettoyé'} | Accuracy : {accuracy_clean:.2%} | F1-Score : {f1_clean:.4f}")
print(f"{'USE + texte brut'} | Accuracy : {accuracy_raw:.2%} | F1-Score : {f1_raw:.4f}")

USE + texte nettoyé | Accuracy : 97.76% | F1-Score : 0.9196
USE + texte brut | Accuracy : 98.12% | F1-Score : 0.9320


Le modèle avec texte brut performe légèrement mieux :
- Accuracy +0.36%
- F1-Score +0.0124

Cela tend à confirmer l'hypothèse : l'USE tire parti des signaux bruts comme les majuscules, la ponctuation et les chiffres (ex: "WIN!!!", "FREE", "£1000") qui sont justement caractéristiques des spams.
Le nettoyage du texte s'avère ici légèrement contre-productif car il efface de l'information pertinente que l'USE sait exploiter.

Avec un F1-Score de 0.9320, le modèle rate peu de spams tout en générant peu de fausses alertes sur les vrais messages. La différence entre les deux modèles reste quand même assez faible.